In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision.io import read_image
from torchvision.transforms import v2 as T
import os
import matplotlib.pyplot as plt
import math
from torch.optim.lr_scheduler import StepLR

# --- 1. Reversible Building Block (Updated) ---
class ReversibleBlock(nn.Module):
    def __init__(self, in_channels, middle_channels=32, dropout_rate=0.2):
        super(ReversibleBlock, self).__init__()
        self.in_channels_half = in_channels // 2


        self.F = nn.Sequential(
            nn.Conv2d(self.in_channels_half, middle_channels, 3, padding=1),
            nn.Tanh(),
            nn.Conv2d(middle_channels, middle_channels, 3, padding=1), # Added convolution
            nn.Tanh(),
            nn.Conv2d(middle_channels, middle_channels, 3, padding=1), # Added convolution
            nn.Tanh(),
            nn.Dropout(dropout_rate), # Added dropout
            nn.Conv2d(middle_channels, self.in_channels_half, 3, padding=1)
        )
        self.G = nn.Sequential(
            nn.Conv2d(self.in_channels_half, middle_channels, 3, padding=1),
            nn.Tanh(),
            nn.Conv2d(middle_channels, middle_channels, 3, padding=1), # Added convolution
            nn.Tanh(),
            nn.Conv2d(middle_channels, middle_channels, 3, padding=1), # Added convolution
            nn.Tanh(),
            nn.Dropout(dropout_rate), # Added dropout
            nn.Conv2d(middle_channels, self.in_channels_half, 3, padding=1)
        )

    def forward(self, x):
        x1, x2 = torch.chunk(x, 2, dim=1)
        y1 = x1 + self.F(x2)
        y2 = x2 + self.G(y1)
        return torch.cat([y1, y2], dim=1)

    def inverse(self, y):
        y1, y2 = torch.chunk(y, 2, dim=1)
        x2 = y2 - self.G(y1)
        x1 = y1 - self.F(x2)
        return torch.cat([x1, x2], dim=1)

# --- 2. Fully Reversible Invertible U-Net (Original) ---
class InvertibleUNet(nn.Module):
    # This model now expects a multi-channel input from the squeeze operation
    def __init__(self, in_channels=5, num_levels=4):
        super(InvertibleUNet, self).__init__()
        self.num_levels = num_levels

        self.encoders = nn.ModuleList()
        self.decoders = nn.ModuleList()

        ch = in_channels
        skip_channels = []
        for i in range(num_levels):
            self.encoders.append(ReversibleBlock(ch * 2))
            skip_channels.append(ch // 2)
            ch *= 2

        self.bottleneck = ReversibleBlock(ch)

        for i in range(num_levels):
            skip_ch = skip_channels.pop()
            decoder_in_ch = (ch // 4) + skip_ch
            self.decoders.insert(0, ReversibleBlock(decoder_in_ch))
            ch = decoder_in_ch

    def squeeze(self, x):
        B, C, H, W = x.shape
        x = x.view(B, C, H // 2, 2, W // 2, 2)
        x = x.permute(0, 1, 3, 5, 2, 4).contiguous()
        return x.view(B, C * 4, H // 2, W // 2)

    def unsqueeze(self, x):
        B, C, H, W = x.shape
        x = x.view(B, C // 4, 2, 2, H, W)
        x = x.permute(0, 1, 4, 2, 5, 3).contiguous()
        return x.view(B, C // 4, H * 2, W * 2)

    def forward(self, x):
        skip_connections = []
        for i in range(self.num_levels):
            x_main, x_skip = torch.chunk(x, 2, dim=1)
            skip_connections.append(x_skip)
            x_main_squeezed = self.squeeze(x_main)
            x = self.encoders[i](x_main_squeezed)

        x = self.bottleneck(x)

        for i in range(self.num_levels):
            x = self.unsqueeze(x)
            x_skip = skip_connections.pop()
            x = torch.cat([x, x_skip], dim=1)
            x = self.decoders[-(i+1)](x)

        return x

    def inverse(self, y):
        skip_connections = []
        for i in range(self.num_levels):
            y = self.decoders[i].inverse(y)
            y_main, y_skip = torch.chunk(y, 2, dim=1)
            skip_connections.append(y_skip)
            y = self.squeeze(y_main)

        y = self.bottleneck.inverse(y)

        for i in range(self.num_levels):
            y = self.encoders[-(i+1)].inverse(y)
            y = self.unsqueeze(y)
            y_skip = skip_connections.pop()
            y = torch.cat([y, y_skip], dim=1)

        return y

# --- 3. Wrapper Model for Squeeze/Unsqueeze and Normalization (Original) ---
class SqueezeNetWrapper(nn.Module):
    def __init__(self):
        super(SqueezeNetWrapper, self).__init__()
        # The inner U-Net now takes 4 channels as input
        self.unet = InvertibleUNet(in_channels=4)

    def squeeze(self, x):
        B, C, H, W = x.shape
        x = x.view(B, C, H // 2, 2, W // 2, 2)
        x = x.permute(0, 1, 3, 5, 2, 4).contiguous()
        return x.view(B, C * 4, H // 2, W // 2)

    def unsqueeze(self, x):
        B, C, H, W = x.shape
        x = x.view(B, C // 4, 2, 2, H, W)
        x = x.permute(0, 1, 4, 2, 5, 3).contiguous()
        return x.view(B, C // 4, H * 2, W * 2)

    def forward(self, x):
        x_norm = (x / 127.5) - 1.0
        x_squeezed = self.squeeze(x_norm)
        modified_squeezed = self.unet(x_squeezed)
        x_final_squeezed = self.unsqueeze(modified_squeezed)
        x_final = (x_final_squeezed + 1.0) * 127.5
        return x_final

    def inverse(self, y):
        y_norm = (y / 127.5) - 1.0
        y_squeezed = self.squeeze(y_norm)
        reconstructed_squeezed = self.unet.inverse(y_squeezed)
        y_final_squeezed = self.unsqueeze(reconstructed_squeezed)
        y_final = (y_final_squeezed + 1.0) * 127.5
        return y_final

# --- 4. Data Handling and Training Logic (Original) ---
class CustomImageDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.img_files = os.listdir(img_dir)
        self.transform = transform

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_files[idx])
        image = read_image(img_path).to(torch.float32)
        if self.transform:
            image = self.transform(image)
        return image

def create_target_image(image, gamma=0.7, zero_width=15):
    boosted_image = 255.0 * ((image / 255.0) ** gamma)
    clipped_image = torch.clamp(boosted_image, min=zero_width, max=255)
    return clipped_image

def calculate_psnr(img1, img2, max_val=255.0):
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0:
        return float('inf')
    return 20 * math.log10(max_val / math.sqrt(mse))

def get_normal_histogram(x, bins=256, min_val=0.0, max_val=255.0):
    x_cpu = x.cpu()
    x_flat = x_cpu.view(-1)
    hist = torch.histc(x_flat, bins=bins, min=min_val, max=max_val)
    return hist.numpy()

def train_model(model, train_loader, val_loader, device, epochs=400, zero_penalty_weight=0.3, zero_width=15):
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        for images in train_loader:
            images = images.to(device)
            optimizer.zero_grad()

            modified_images = model(images)
            target_images = create_target_image(images, zero_width=zero_width)

            # Main MSE Loss
            mse_loss = criterion(modified_images, target_images)

            # --- CORRECTED: Penalty for pixels in forbidden zones ---
            # Penalize values below the lower bound
            lower_penalty = torch.relu(zero_width - modified_images).mean()
            # Penalize values above the upper bound
            upper_penalty = torch.relu(modified_images - 255).mean()

            boundary_penalty = lower_penalty + (10*upper_penalty)

            # Combine the losses
            total_loss = mse_loss + zero_penalty_weight * boundary_penalty

            total_loss.backward()
            optimizer.step()
            total_train_loss += total_loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {total_train_loss / len(train_loader):.6f}")

        model.eval()
        with torch.no_grad():
            total_val_loss = 0
            for images in val_loader:
                images = images.to(device)
                modified_images = model(images)
                target_images = create_target_image(images, zero_width=zero_width)

                mse_loss = criterion(modified_images, target_images)
                lower_penalty = torch.relu(zero_width - modified_images).mean()
                upper_penalty = torch.relu(modified_images - 255).mean()
                boundary_penalty = lower_penalty + (10*upper_penalty)
                total_loss = mse_loss + zero_penalty_weight * boundary_penalty
                total_val_loss += total_loss.item()
            print(f"Epoch {epoch+1}/{epochs}, Val Loss: {total_val_loss / len(val_loader):.6f}")
        checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': total_val_loss,
        'train_loss': total_train_loss
        # Add other relevant information here
        }
        if epoch % 10 == 0:
            torch.save(checkpoint, f'/content/drive/MyDrive/checkpoint_epoch_{epoch}.pth')

if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    IMG_DIR = '/content/drive/MyDrive/ILSVRC2013_train_extra9'

    if not os.path.exists(IMG_DIR) or not os.listdir(IMG_DIR):
        print(f"Error: Directory is empty or not found at '{IMG_DIR}'")
        print("Please create this directory and place your 1000 images inside it.")
    else:
        transform = T.Compose([
            T.Grayscale(),
            T.Resize((256, 256), antialias=True)
        ])

        full_dataset = CustomImageDataset(img_dir=IMG_DIR, transform=transform)
        train_size = int(0.8 * len(full_dataset))
        val_size = len(full_dataset) - train_size
        train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

        train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)

        print("\n--- Starting Model Training ---")
        model = SqueezeNetWrapper().to(device)

        train_model(model, train_dataloader, val_dataloader, device, epochs=400)

        print("\n--- Saving Final Model and Visualizing Results ---")

        torch.save(model.state_dict(), 'final_squeeze_unet.pth')
        print("Model saved to final_squeeze_unet.pth")

        model.eval()
        with torch.no_grad():
            original_image = next(iter(val_dataloader))[0].unsqueeze(0).to(device)
            modified_image = model(original_image)
            reconstructed_image = model.inverse(modified_image)

            psnr_modified = calculate_psnr(original_image, modified_image)
            psnr_reconstructed = calculate_psnr(original_image, reconstructed_image)
            mae_reconstruction = torch.mean(torch.abs(original_image - reconstructed_image))

            print(f"\nPSNR (Original vs. Modified): {psnr_modified:.2f} dB")
            print(f"PSNR (Original vs. Reconstructed): {psnr_reconstructed:.2f} dB")
            print(f"Mean Absolute Reconstruction Error: {mae_reconstruction.item():.6f}")

            original_image_cpu = original_image.squeeze().cpu().numpy()
            modified_image_cpu = modified_image.squeeze().cpu().numpy()
            reconstructed_image_cpu = reconstructed_image.squeeze().cpu().numpy()

            plt.figure(figsize=(18, 6))
            plt.subplot(1, 3, 1); plt.imshow(original_image_cpu, cmap='gray', vmin=0, vmax=255); plt.title("Original Image"); plt.axis('off')
            plt.subplot(1, 3, 2); plt.imshow(modified_image_cpu, cmap='gray', vmin=0, vmax=255); plt.title("Modified Image"); plt.axis('off')
            plt.subplot(1, 3, 3); plt.imshow(reconstructed_image_cpu, cmap='gray', vmin=0, vmax=255); plt.title("Reconstructed Image"); plt.axis('off')
            plt.show()

            original_hist = get_normal_histogram(original_image)
            modified_hist = get_normal_histogram(modified_image)
            reconstructed_hist = get_normal_histogram(reconstructed_image)

            plt.figure(figsize=(18, 5))
            plt.subplot(1, 3, 1); plt.bar(range(256), original_hist); plt.title("Original Histogram")
            plt.subplot(1, 3, 2); plt.bar(range(256), modified_hist); plt.title("Modified Histogram")
            plt.subplot(1, 3, 3); plt.bar(range(256), reconstructed_hist); plt.title("Reconstructed Histogram")
            plt.show()


Using device: cuda

--- Starting Model Training ---
Epoch 1/400, Train Loss: 74.473882
Epoch 1/400, Val Loss: 22.045187
Epoch 2/400, Train Loss: 25.520558
Epoch 2/400, Val Loss: 15.918365
Epoch 3/400, Train Loss: 19.672935
Epoch 3/400, Val Loss: 14.284034
Epoch 4/400, Train Loss: 16.665490
Epoch 4/400, Val Loss: 12.096049
Epoch 5/400, Train Loss: 14.624437
Epoch 5/400, Val Loss: 11.127562
Epoch 6/400, Train Loss: 13.128274
Epoch 6/400, Val Loss: 10.198750
Epoch 7/400, Train Loss: 12.025252
Epoch 7/400, Val Loss: 10.150931
Epoch 8/400, Train Loss: 11.051840
Epoch 8/400, Val Loss: 8.497060
Epoch 9/400, Train Loss: 10.387728
Epoch 9/400, Val Loss: 7.982860
Epoch 10/400, Train Loss: 9.784975
Epoch 10/400, Val Loss: 7.750009
Epoch 11/400, Train Loss: 9.310018
Epoch 11/400, Val Loss: 7.394822
Epoch 12/400, Train Loss: 9.017314
Epoch 12/400, Val Loss: 7.360343
Epoch 13/400, Train Loss: 8.627380
Epoch 13/400, Val Loss: 7.047812
Epoch 14/400, Train Loss: 8.401268
Epoch 14/400, Val Loss: 6.91186

In [ ]:
transform = T.Compose([
            T.Grayscale(),
            T.Resize((256, 256), antialias=True)
        ])

In [ ]:
import numpy as np

def find_histogram_peaks(histogram):
    """
    Input:
        histogram: A list or numpy array of length 256 representing pixel value frequencies.
    Output:
        main_peak: Index of the bin with the maximum frequency.
        left_peak: Index of the nearest local peak to the left of the main peak.
        right_peak: Index of the nearest local peak to the right of the main peak.
    """
    histogram = np.array(histogram).astype(np.uint8)
    main_peak = np.argmax(histogram)

    # Search for left peak: highest bin before main_peak
    left_peak = None
    for i in range(main_peak - 1, 0, -1):
        if histogram[i] > histogram[i - 1] and histogram[i] > histogram[i + 1]:
            left_peak = i
            break

    # Search for right peak: highest bin after main_peak
    right_peak = None
    for i in range(main_peak + 1, len(histogram) - 1):
        if histogram[i] > histogram[i - 1] and histogram[i] > histogram[i + 1]:
            right_peak = i
            break

    # Fallbacks if local peaks aren't found
    if left_peak is None:
        left_peak = max(0, main_peak - 1)
    if right_peak is None:
        right_peak = min(255, main_peak + 1)

    return main_peak, left_peak, right_peak

In [ ]:
import torch
import torch.nn.functional as F

def gaussian(window_size, sigma):
    gauss = torch.Tensor([math.exp(-(x - window_size//2)**2/float(2*sigma**2)) for x in range(window_size)])
    return gauss/gauss.sum()

def create_window(window_size, channel):
    _1D_window = gaussian(window_size, 1.5).unsqueeze(1)
    _2D_window = _1D_window.mul(_1D_window.t()).unsqueeze(0).unsqueeze(0)
    window = _2D_window.expand(channel, 1, window_size, window_size).contiguous()
    return window

def ssim(img1, img2, window_size=11, size_average=True, max_val=255):
    channel = img1.size(-3)
    window = create_window(window_size, channel)

    if img1.is_cuda:
        window = window.cuda(img1.get_device())
    window = window.type_as(img1)

    mu1 = F.conv2d(img1, window, padding=window_size//2, groups=channel)
    mu2 = F.conv2d(img2, window, padding=window_size//2, groups=channel)

    mu1_sq = mu1.pow(2)
    mu2_sq = mu2.pow(2)
    mu1_mu2 = mu1 * mu2

    sigma1_sq = F.conv2d(img1 * img1, window, padding=window_size//2, groups=channel) - mu1_sq
    sigma2_sq = F.conv2d(img2 * img2, window, padding=window_size//2, groups=channel) - mu2_sq
    sigma12 = F.conv2d(img1 * img2, window, padding=window_size//2, groups=channel) - mu1_mu2

    C1 = (0.01 * max_val) ** 2
    C2 = (0.03 * max_val) ** 2

    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))

    if size_average:
        return ssim_map.mean()
    else:
        return ssim_map.mean(1).mean(1).mean(1)